# Прогноз суточной аренды Capital Bikeshare

Ряд не из занятий: не влажность, не Microsoft, не WTI, не Favorita.

Берём дневную аренду велосипедов в Вашингтоне, 2011–2012. Есть рост от первого года ко второму и два цикла: неделя и год.

Страница набора: https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset

Архив: https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip



## Подготовка


In [ ]:
%pip -q install prophet statsforecast statsmodels


In [ ]:
import io
import zipfile
import urllib.request
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX

import prophet as fp
from prophet.diagnostics import cross_validation

from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, SeasonalNaive, MSTL, AutoTheta, HoltWinters
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae, rmse


In [ ]:
def adf_test(timeseries):
    print("Results of Dickey-Fuller Test:")
    dftest = adfuller(timeseries, autolag="AIC")
    dfoutput = pd.Series(
        dftest[0:4],
        index=["Test Statistic", "p-value", "#Lags Used", "Number of Observations Used"],
    )
    for key, value in dftest[4].items():
        dfoutput["Critical Value (%s)" % key] = value
    print(dfoutput)


In [ ]:
def kpss_test(timeseries):
    print("Results of KPSS Test:")
    kpsstest = kpss(timeseries, regression="c", nlags="auto")
    kpss_output = pd.Series(
        kpsstest[0:3], index=["Test Statistic", "p-value", "Lags Used"]
    )
    for key, value in kpsstest[3].items():
        kpss_output["Critical Value (%s)" % key] = value
    print(kpss_output)


In [ ]:
def forecast_errors(actual, predicted):
    actual, predicted = np.asarray(actual), np.asarray(predicted)
    error = actual - predicted
    nonzero = actual != 0
    mape = np.mean(np.abs(error[nonzero] / actual[nonzero])) * 100 if nonzero.any() else np.nan
    return {'MAE': np.abs(error).mean(),
            'RMSE': np.sqrt(np.mean(error ** 2)),
            'WAPE(%)': np.abs(error).sum() / np.abs(actual).sum() * 100,
            'MAPE(%)': mape}


## Данные

Ячейка скачивает архив UCI и достаёт `day.csv`. Повторный запуск архив не качает.



In [ ]:
url = 'https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip'
dest = Path('bike_day.csv')
if not dest.exists():
    raw_zip = urllib.request.urlopen(url, timeout=60).read()
    with zipfile.ZipFile(io.BytesIO(raw_zip)) as zf:
        dest.write_bytes(zf.read('day.csv'))
print(dest.resolve(), dest.stat().st_size)


In [ ]:
raw = pd.read_csv('bike_day.csv', parse_dates=['dteday'])
raw.head()


`cnt` — число аренд за день. Температура в файле нормирована: −8 °C это 0, 39 °C это 1. Перевожу в градусы. Праздник уже размечен.



In [ ]:
raw = raw.rename(columns={'dteday': 'ds'}).sort_values('ds')
raw['temp_c'] = raw['temp'] * 47 - 8
raw['y'] = raw['cnt']
raw = raw.set_index('ds').asfreq('D')
print(raw.shape)
print(raw.index.min().date(), raw.index.max().date())
raw[['y', 'temp_c', 'holiday', 'workingday']].describe().round(2)


In [ ]:
fig = px.line(raw, y='y', title='Суточная аренда Capital Bikeshare')
fig.update_layout(template='plotly_white', width=1000, height=450)
fig.update_xaxes(title_text='Дата')
fig.update_yaxes(title_text='Велосипедов')
fig.show()


## Train и test

Режем по времени, 80/20. Хвост — будущее, случайно строки не мешаем.



In [ ]:
split = int(len(raw) * 0.8)
train = raw.iloc[:split].copy()
test = raw.iloc[split:].copy()
print('train', len(train), train.index.min().date(), train.index.max().date())
print('test ', len(test), test.index.min().date(), test.index.max().date())

fig = px.line(title='Обучающая и тестовая выборки')
fig.add_scatter(x=train.index, y=train['y'], mode='lines', name='train')
fig.add_scatter(x=test.index, y=test['y'], mode='lines', name='test')
fig.update_layout(template='plotly_white', width=1000, height=450)
fig.update_xaxes(title_text='Дата')
fig.update_yaxes(title_text='Велосипедов')
fig.show()


Train — 584 дня, по 6 августа 2012. Test — 147 дней, с 7 августа по 31 декабря 2012. На графике виден рост 2012 года и провал к зиме.



## Стационарность


In [ ]:
adf_test(train['y'])
print('---')
kpss_test(train['y'])


In [ ]:
adf_test(train['y'].diff().dropna())
print('---')
kpss_test(train['y'].diff().dropna())


Уровень по KPSS нестационарен. Одна разность стационарна по обоим тестам, поэтому в SARIMAX `d = 1`. Сезонный период для SARIMAX — 7: неделя видна на буднях и выходных. Год в SARIMAX не ставлю, двух лет на сезон 365 мало. Год отдают Prophet и MSTL.



## Регрессоры

Dummy — `holiday`. Дата праздника известна на январь 2013 заранее.

Непрерывный — `temp_c`. Тёплая погода связана с большим спросом. На будущие 30 дней фактической температуры нет, дальше берём сценарий «как в тот же день прошлого года».



In [ ]:
print('корреляция, train:', train['temp_c'].corr(train['y']).round(3))
print('праздник:', train.loc[train.holiday == 1, 'y'].mean().round(1))
print('обычный день:', train.loc[train.holiday == 0, 'y'].mean().round(1))

fig = px.scatter(train, x='temp_c', y='y', title='Train: температура и аренда', template='plotly_white')
fig.update_layout(width=700, height=450)
fig.update_xaxes(title_text='Температура, C')
fig.update_yaxes(title_text='Велосипедов')
fig.show()


Корреляция на train около 0.70. В праздник средняя аренда ниже, но праздников мало. На диаграмме ниже 5 °C точки лежат у 1000–2000, выше 20 °C — у 4000–7000.



## SARIMAX

Порядок задаю вручную, как в statsmodels: `(1, 1, 1) x (1, 0, 1, 7)`. Рядом та же модель без колонок. Остатки смотрю по ACF и Ljung-Box.



In [ ]:
exog_cols = ['holiday', 'temp_c']
sarimax = SARIMAX(
    train['y'], order=(1, 1, 1), seasonal_order=(1, 0, 1, 7),
    exog=train[exog_cols],
    enforce_stationarity=False, enforce_invertibility=False,
).fit(disp=False)
sarima = SARIMAX(
    train['y'], order=(1, 1, 1), seasonal_order=(1, 0, 1, 7),
    enforce_stationarity=False, enforce_invertibility=False,
).fit(disp=False)
print(sarimax.summary())


In [ ]:
resid = sarimax.resid.dropna()
print(acorr_ljungbox(resid, lags=[7, 14, 28], model_df=4, return_df=True))
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
plot_acf(resid, lags=28, ax=ax[0])
plot_pacf(resid, lags=28, ax=ax[1])
plt.tight_layout()
plt.show()


Температура значима, около +86 велосипедов на градус. Праздник около −12, p = 0.96. Ljung-Box на лаге 7 ещё значим, на 14 и 28 уже нет: неделя забрана не до конца.



## Формат StatsForecast

Как в лекции 9–10: столбцы `unique_id`, `ds`, `y`. AutoARIMA здесь играет роль SARIMAX и считается с колонками и без них.



In [ ]:
def to_sf(data):
    out = data.reset_index().rename(columns={'ds': 'ds'})
    out['unique_id'] = 'bikes'
    return out

train_sf = to_sf(train)[['unique_id', 'ds', 'y']]
test_sf = to_sf(test)[['unique_id', 'ds', 'y']]
train_x = to_sf(train)[['unique_id', 'ds', 'y', 'holiday', 'temp_c']]
test_x = to_sf(test)[['unique_id', 'ds', 'holiday', 'temp_c']]

arima_options = dict(season_length=7, max_p=2, max_q=2, max_P=1, max_Q=1)
sf_sarima = StatsForecast(models=[AutoARIMA(**arima_options)], freq='D')
fc_sarima = sf_sarima.forecast(df=train_sf, h=len(test))
sf_sarimax = StatsForecast(models=[AutoARIMA(**arima_options)], freq='D')
fc_sarimax = sf_sarimax.forecast(df=train_x, h=len(test), X_df=test_x)
print(forecast_errors(test['y'], fc_sarima['AutoARIMA']))
print(forecast_errors(test['y'], fc_sarimax['AutoARIMA']))


## Prophet

Сетка короче, чем на лекции: два значения излома тренда и два значения силы сезона. Каждую пару проверяю на последних 60 днях train, test не трогаю.



In [ ]:
val_h = 60
y_tr, y_val = train.iloc[:-val_h], train.iloc[-val_h:]
param_grid = {
    'changepoint_prior_scale': [0.05, 0.2],
    'seasonality_prior_scale': [1.0, 10.0],
}
all_params = [dict(zip(param_grid, values)) for values in itertools.product(*param_grid.values())]
tuning_rows = []
for params in all_params:
    model = fp.Prophet(weekly_seasonality=True, yearly_seasonality=True, daily_seasonality=False, **params)
    model.add_regressor('holiday')
    model.add_regressor('temp_c')
    frame = y_tr.reset_index()
    model.fit(frame[['ds', 'y', 'holiday', 'temp_c']])
    fut = y_val.reset_index()
    pred = model.predict(fut[['ds', 'holiday', 'temp_c']])['yhat']
    tuning_rows.append({**params, 'MAE': np.abs(y_val['y'] - pred).mean()})
tuning_results = pd.DataFrame(tuning_rows).sort_values('MAE')
tuning_results


In [ ]:
best_params = tuning_results.iloc[0][list(param_grid)].to_dict()
best_params

def fit_prophet(data, use_x):
    model = fp.Prophet(weekly_seasonality=True, yearly_seasonality=True, daily_seasonality=False, **best_params)
    frame = data.reset_index()
    cols = ['ds', 'y']
    if use_x:
        model.add_regressor('holiday')
        model.add_regressor('temp_c')
        cols += ['holiday', 'temp_c']
    model.fit(frame[cols])
    return model

prophet_x = fit_prophet(train, True)
prophet_0 = fit_prophet(train, False)


Лучшая пара — `changepoint_prior_scale=0.05`, `seasonality_prior_scale=10`. Соседние пары отличаются меньше чем на 6 велосипедов. Тренд лучше держать гладким.



## MSTL, AutoTheta, Holt–Winters

Как в лекции 9–10. MSTL снимает неделю и год. AutoTheta и Holt–Winters — ещё две статистические модели, сезон 7. Регрессоры они не принимают, сравнение с колонками и без для них не делаю.



In [ ]:
models = [
    MSTL(season_length=[7, 365], trend_forecaster=AutoARIMA(max_p=2, max_q=2, seasonal=False)),
    AutoTheta(season_length=7),
    HoltWinters(season_length=7, error_type='A'),
    SeasonalNaive(season_length=7),
]
sf_pipeline = StatsForecast(models=models, freq='D')
pipeline_test = sf_pipeline.forecast(df=train_sf, h=len(test))
pipeline_test.head()


## Кросс-валидация

`StatsForecast.cross_validation` сама режет окна. Горизонт 30 дней, шаг 30, три окна. Это выбор на train.



In [ ]:
crossvalidation_df = sf_pipeline.cross_validation(
    df=train_sf, h=30, step_size=30, n_windows=3,
)
model_names = [c for c in crossvalidation_df.columns if c not in ('unique_id', 'ds', 'cutoff', 'y')]
evaluate(crossvalidation_df, metrics=[mae, rmse], models=model_names)


In [ ]:
# SARIMAX и Prophet с колонками и без, те же три начала
h = 30
origins = [int(len(train) * p) for p in (0.7, 0.8, 0.9)]
cv_rows = []
for origin in origins:
    tr = train.iloc[:origin]
    va = train.iloc[origin:origin + h]
    sx = SARIMAX(tr['y'], order=(1, 1, 1), seasonal_order=(1, 0, 1, 7), exog=tr[exog_cols], enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    s0 = SARIMAX(tr['y'], order=(1, 1, 1), seasonal_order=(1, 0, 1, 7), enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    cv_rows.append({'Модель': 'SARIMAX', **forecast_errors(va['y'], sx.get_forecast(h, exog=va[exog_cols]).predicted_mean)})
    cv_rows.append({'Модель': 'SARIMA', **forecast_errors(va['y'], s0.get_forecast(h).predicted_mean)})
    px = fit_prophet(tr, True)
    p0 = fit_prophet(tr, False)
    fut = va.reset_index()
    cv_rows.append({'Модель': 'Prophet с колонками', **forecast_errors(va['y'], px.predict(fut[['ds', 'holiday', 'temp_c']])['yhat'])})
    cv_rows.append({'Модель': 'Prophet', **forecast_errors(va['y'], p0.predict(fut[['ds']])['yhat'])})
cv = pd.DataFrame(cv_rows)
cv.groupby('Модель')[['MAE', 'RMSE', 'WAPE(%)']].mean().sort_values('MAE').round(1)


Метрика выбора — MAE: ошибка в велосипедах, нулевых дней нет. Средний MAE на трёх окнах train: SARIMAX 903, SARIMA 927, Prophet с колонками 980, Prophet 1009. Колонки улучшают обе модели, которые их принимают. По train лучший — SARIMAX.



## Test


In [ ]:
fc_sx = sarimax.get_forecast(len(test), exog=test[exog_cols]).predicted_mean
fc_s = sarima.get_forecast(len(test)).predicted_mean
fut = test.reset_index()
fc_px = prophet_x.predict(fut[['ds', 'holiday', 'temp_c']])['yhat']
fc_p0 = prophet_0.predict(fut[['ds']])['yhat']

rows = [
    {'Модель': 'SARIMAX', **forecast_errors(test['y'], fc_sx)},
    {'Модель': 'SARIMA', **forecast_errors(test['y'], fc_s)},
    {'Модель': 'Prophet с колонками', **forecast_errors(test['y'], fc_px)},
    {'Модель': 'Prophet', **forecast_errors(test['y'], fc_p0)},
]
for col in pipeline_test.columns:
    if col in ('unique_id', 'ds'):
        continue
    rows.append({'Модель': col, **forecast_errors(test['y'], pipeline_test[col])})
pd.DataFrame(rows).set_index('Модель').sort_values('MAE').round(1)


In [ ]:
fig = px.line(title='Test: факт и прогнозы')
fig.add_scatter(x=test.index, y=test['y'], mode='lines', name='факт')
fig.add_scatter(x=test.index, y=fc_sx, mode='lines', name='SARIMAX')
fig.add_scatter(x=test.index, y=fc_px, mode='lines', name='Prophet с колонками')
fig.add_scatter(x=test.index, y=pipeline_test['MSTL'], mode='lines', name='MSTL')
fig.update_layout(template='plotly_white', width=1000, height=450)
fig.show()


На test MAE: Prophet с колонками 1054, Prophet 1087, MSTL 1126, SARIMAX 1170, SARIMA 1516. Выбранный по train SARIMAX колонки отрабатывает: без них 1516. Первым на test он не становится. Осенний спад лучше берёт Prophet, у него есть год. Test в выборе не участвует.



## Прогноз на 30 дней

Лучшую по кросс-валидации модель учу на всём ряде. Праздник 1 января 2013 известен. Температуру беру с того же дня 2012 года: это сценарий, не прогноз погоды.



In [ ]:
future_idx = pd.date_range(raw.index.max() + pd.Timedelta(days=1), periods=30, freq='D')
temp_scenario = []
for day in future_idx:
    same = raw.loc[raw.index.strftime('%m-%d') == day.strftime('%m-%d'), 'temp_c']
    temp_scenario.append(same.iloc[-1])
X_future = pd.DataFrame({
    'holiday': [int(day == pd.Timestamp('2013-01-01')) for day in future_idx],
    'temp_c': temp_scenario,
}, index=future_idx)

full = SARIMAX(
    raw['y'], order=(1, 1, 1), seasonal_order=(1, 0, 1, 7),
    exog=raw[exog_cols],
    enforce_stationarity=False, enforce_invertibility=False,
).fit(disp=False)
future = full.get_forecast(30, exog=X_future).predicted_mean

fig = px.line(title='Прогноз на январь 2013')
fig.add_scatter(x=raw.index[-90:], y=raw['y'].iloc[-90:], mode='lines', name='факт, хвост')
fig.add_scatter(x=future.index, y=future, mode='lines', name='SARIMAX')
fig.update_layout(template='plotly_white', width=1000, height=450)
fig.show()
future.round(0).head(10)


## Почему эта модель

На окнах train SARIMAX дала наименьший MAE, 903. Температура значима, разность снимает рост 2011–2012, неделя задана явно. Колонки полезны и у SARIMAX, и у Prophet, и у AutoARIMA.

На test она не первая: год у Prophet и MSTL, у SARIMAX сезон только 7. Решение принято до test, поэтому на январь 2013 оставляю SARIMAX.

Прогноз лежит около 2300–3600 велосипедов в день. Ограничения: праздник незначим, лаг 7 в остатках ещё есть, температура будущего — сценарий 2012 года. Если январь 2013 холоднее, прогноз завысит спрос примерно на 86 велосипедов на каждый лишний градус.

